## HDSoCv1_evalr2 ROI Mode

This notebook demostrates how to use ROI mode. Still experimental and parameters need to be explored.

### Naludaq Version
*Min Version*: `0.36.0`

### Compatible Boards
+ `HDSoCv1_evalr2`

## Imports

In [1]:
%load_ext autoreload
%autoreload 2
import time
import itertools
import numpy as np
from copy import deepcopy
from pathlib import Path


from naludaq.board import Board, startup_board
from naludaq.communication import AnalogRegisters, DigitalRegisters, ControlRegisters
from naludaq.parsers import get_parser
from naludaq.tools import get_pedestals_controller
from naludaq.tools.pedestals.pedestals_correcter import PedestalsCorrecter

from naludaq.backend import AcquisitionManager, ConnectionManager, RemoteAcquisition

import matplotlib.pyplot as plt

D3XX import failed: Support for USB 3 on Linux is unavailable


In [2]:
# import logging


# def setup_logger(level=logging.DEBUG):
#     """Setup a basic logger."""
#     logger = logging.getLogger()
#     handler = logging.StreamHandler()
#     handler.setFormatter(
#         logging.Formatter("%(asctime)s %(name)-30s [%(levelname)-6s]: %(message)s")
#     )
#     logger.addHandler(handler)
#     logger.setLevel(level)
#     suppress = [
#         "naludaq.UART",
#         "naludaq.FTDI",
#     ]
#     # for name in suppress:
#     #     logging.getLogger(name).setLevel(logging.CRITICAL)
#     return logger


# try:
#     logger.debug("logger already setup")
# except:
#     logger = setup_logger()

## Function Definitions

In [3]:
def create_acquisition(name: str, board: Board, readout_metadata: dict = None) -> RemoteAcquisition:
    """Create an acquisition with the given name and optional readout metadata.
    Sets the output to the created acquisition and returns the acquisition object.
        Args:
            name (str): The name of the acquisition to be created.
            board (Board): The board object to be used for creating the acquisition.
            readout_metadata (dict, optional): A dictionary containing metadata for the readout. Defaults to None.
        Returns:
            RemoteAcquisition: The created acquisition object with the specified name and readout metadata.
    """
    am = AcquisitionManager(board)
    acquisition = am.create(name)
    acquisition.set_output()
    if readout_metadata:
        acquisition.readout_metadata = readout_metadata
    return acquisition

def parse_data(raw_data: dict, board: Board, correct_peds: bool = True) -> dict:
    """Parses an event dict that contains a 'rawdata' key.
        Args:
            raw_data (dict): A dictionary containing a 'rawdata' key with the raw data to be parsed.
            board (Board): The board object containing the parameters and pedestals for parsing.
            correct_peds (bool, optional): Whether to apply pedestal correction to the parsed event. Defaults to True.
        Returns:
            dict: A dictionary representing the parsed event.
    """
    PARSER = get_parser(board.params)
    evt = PARSER.parse(raw_data)
    if correct_peds:
        PC = PedestalsCorrecter(board.params, board.pedestals)
        PC.run(evt)
    return evt

def create_acq_name(base_name: str) -> str:
    """Create an acquisition name by appending a timestamp to the base name.
        Args:
            base_name (str): The base name to which the timestamp will be appended.
        Returns:
            str: The generated acquisition name with the timestamp in the form <base_name>_<timestamp>.
    """
    timestr = time.strftime("%Y-%m-%d-%H-%M-%S")
    return base_name + "_" + timestr

def clear_figs():
    """Clears all matplotlib figures"""
    plt.close("all")

def create_fig(figsize: tuple) -> plt.Figure:
    """Creates a matplotlib figure with the given size, constrained layout, and white background.
        Args:
            figsize (tuple): A tuple specifying the width and height of the figure.
        Returns:
            plt.Figure: A matplotlib figure object with the specified properties.
    """

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    fig.set_facecolor('white')
    return fig

def plot_single_event(event, show: bool = True):
    """Plots a single event to the current matplotlib figure. Each channel is plotted separately with markers and a legend.
        Args:
            event (dict): A dictionary representing the event to be plotted.
            show (bool, optional): Whether to display the plot. Defaults to True.
    """
    for chan, chan_data in enumerate(event["data"]):
        if isinstance(chan_data, np.ndarray) and chan_data.size == 0:
            continue
        if len(chan_data) == 0:
            continue
        if np.all(np.isnan(chan_data)):
            continue
        plt.plot(chan_data, '.-', markersize=10, label=f"Channel {chan}")
    plt.legend(fontsize=12)
    if show:
        plt.show()


## Directory Setup

In [4]:
DATA_DIR = Path("./data").absolute()
DATA_DIR.mkdir(exist_ok=True)
SERVER_DIR = Path("./server").absolute()
SERVER_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("figures").absolute()
FIGURES_DIR.mkdir(exist_ok=True)

In [6]:
import sys
import naludaq

print("Python:", sys.executable)
print("Python version:", sys.version)
print("naludaq:", naludaq.__version__)

Python: /home/pixadmin/naluenv311/bin/python
Python version: 3.11.13 (main, Jun  1 2026, 00:00:00) [GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]
naludaq: 0.36.1


## Board Initialization

In [8]:
# Board initialization
try:
    BOARD.disconnect()
except:
    pass

BOARD = Board("hdsocv1_evalr2")
BOARD.start_server(str(SERVER_DIR))

BOARD_ADDRESS = ("192.168.1.59", 4660)
RECEIVER_ADDRESS = ("192.168.1.101", 4665)
SERVER_ADDRESS = BOARD.context.address

BOARD.connect_udp(
    BOARD_ADDRESS,
    RECEIVER_ADDRESS,
    SERVER_ADDRESS
)

BOARD.initialize()

2026-09-23T22:32:16.446183Z  INFO ThreadId(44) src/web_api/mod.rs:41: Creating server
2026-09-23T22:32:16.446864Z  INFO ThreadId(44) src/web_api/mod.rs:94: Running server on 127.0.0.1:43729
2026-09-23T22:32:16.446879Z  INFO ThreadId(44) run: src/workers/mod.rs:82: Starting workers
2026-09-23T22:32:16.459988Z DEBUG ThreadId(60) run:run_packager_task_controller: src/workers/packager.rs:206: Starting packager: PackagerConfig { model: "hdsocv1_evalr2", event_stop_word: [250, 90], answer_stop_word: [202, 254] }
2026-09-23T22:32:16.460014Z DEBUG ThreadId(60) run_packager_impl: src/workers/packager.rs:256: Running package worker with config: PackagerConfig { model: "hdsocv1_evalr2", event_stop_word: [250, 90], answer_stop_word: [202, 254] }
2026-09-23T22:32:16.460020Z DEBUG ThreadId(60) run_packager_impl: src/workers/packager.rs:259: SELECTING HDSOC PACKAGER
2026-09-23T22:32:16.462627Z DEBUG ThreadId(60) run:run_packager_task_controller: src/workers/packager.rs:206: Starting packager: Package

ConnectionError: [Status 400, ID 5] failed to open a connection

## Signal Generator

### Signal Generator Settings

In [6]:
PULSE_FREQ = 20
PULSE_WIDTH = 10e-9
PULSE_AMP = 0.15

# If using bursted pulses
N_CYCLES = 10
BURST_PERIOD = 1

### Signal Generator Setup

In [ ]:
# This is using NaluInstruments which is internal to Nalu Scientific.
# Use the appropriate instrument control code for your signal generator.
from naluinstruments.Instruments import Siglent_FuncGen
import time

CHANNEL_INPUT = Siglent_FuncGen(1, "192.168.1.189")

# REGULAR PULSE
CHANNEL_INPUT.set_burst("OFF")
CHANNEL_INPUT.set_wavetype("PULSE")
CHANNEL_INPUT.set_width(PULSE_WIDTH)
CHANNEL_INPUT.set_rise(2e-9)
CHANNEL_INPUT.set_amp(PULSE_AMP)
CHANNEL_INPUT.set_freq(PULSE_FREQ)

# BURST PULSE
# CHANNEL_INPUT.set_wavetype("PULSE")
# CHANNEL_INPUT.set_width(PULSE_WIDTH)
# CHANNEL_INPUT.set_rise(2e-9)
# CHANNEL_INPUT.set_amp(PULSE_AMP)
# CHANNEL_INPUT.set_freq(PULSE_FREQ)

# CHANNEL_INPUT.set_burst("ON")
# CHANNEL_INPUT.set_burst_mode("NCYC")
# CHANNEL_INPUT.set_burst_time(N_CYCLES)
# CHANNEL_INPUT.set_burst_period(1)

print(CHANNEL_INPUT.get_output_settings())

## Data Capture Configuration

In [7]:
DAC_LEVEL = 1804 # (Default DAC Counts for HDSoCv2) Adjust depending on your board.

# You may have to adjust these parameters to center the pulse within the captured data.
WINDOWS = 24
LOOKBACK = 50
WAT = 33
READOUTCHANNELS = 57

ENABLED_READOUT_CHANNELS = [0, 1, 2, 3]

READOUT_TIME = 5

# External Trigger, Trigger Relative Mode
READOUT_PARAMS = {
    "trig": "ext",
    "lb": "trigrel",
    "acq": "raw",
    "ped": "zero",
    "readoutEn": True,
    "singleEv": False,
}

READ_WINDOW = {
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
}

# Created to save with the acquisition metadata for reference when analyzing the data and plotting.
PARAMS = {
    "dac_level": DAC_LEVEL,
    "windows": WINDOWS,
    "lookback": LOOKBACK,
    "write_after_trig": WAT,
    "readoutchannels": READOUTCHANNELS,
    "n_cycle": N_CYCLES,
    "burst_period": BURST_PERIOD,
    "pulse_width": PULSE_WIDTH,
    "pulse_freq": PULSE_FREQ,
    "readout_time": READOUT_TIME,
    "readout_channels": ENABLED_READOUT_CHANNELS,
}

AR = AnalogRegisters(BOARD)
DR = DigitalRegisters(BOARD)
AM = AcquisitionManager(BOARD)

### Set Board Configurations

In [8]:
BOARD.ext_bias.set_dacs(DAC_LEVEL)

2026-09-17T03:06:16.974776Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [AF, 11, 00, 00]
2026-09-17T03:06:16.974814Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [AF, 0F, 43, 92]
2026-09-17T03:06:16.974828Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [AF, 12, 00, 31]
2026-09-17T03:06:16.974842Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [AF, 13, 70, C0]
2026-09-17T03:06:16.974855Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [AF, 10, 00, 00]
2026-09-17T03:06:16.975589Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381: Sending command: [CA, 00, 00, 00]
2026-09-17T03:06:16.977295Z DEBUG ThreadId(04) run:run_connection_task_controller: src/workers/connection.rs:381

### Generate / Load Pedestals

In [9]:
PC = get_pedestals_controller(BOARD, num_captures=10, num_warmup_events=10)
PC.generate_pedestals()

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

### Setup Triggers
Set triggers accordingly to your threshold sweep.

In [ ]:
trigger_values = [1] * BOARD.channels
trigger_values[1] = 240
BOARD.trigger_values = trigger_values
BOARD.trigger.write_triggers()

BOARD.trigger.update_reference_voltages("left", 6, 15)
BOARD.trigger.update_reference_voltages("right", 6, 15)
AR.write("tsel_left", 1)
AR.write("tsel_right", 1)
AR.write("tsgn_left", 1)
AR.write("tsgn_right", 1)

### Set Readout Settings

In [ ]:
BOARD.readout.set_read_window(**READ_WINDOW)
BOARD.readout.set_readout_channels(ENABLED_READOUT_CHANNELS)

BOARD.readout.set_read_window(WINDOWS, LOOKBACK, WAT)
DR.write("readoutchannels", READOUTCHANNELS)

acq_name = create_acq_name("hdsocv1_evalr2_ROI")
acq = create_acquisition(acq_name, BOARD)



### Create an Acquisition

In [ ]:
ACQ_NAME = f"{BOARD.model}_ROI_windows{WINDOWS}"
acq_name = create_acq_name(ACQ_NAME)
readout_metadata = {"params": deepcopy(PARAMS)}
acq = create_acquisition(acq_name, BOARD, readout_metadata)
acq.pedestals = BOARD.pedestals

### Readout Data

In [ ]:
CHANNEL_INPUT.rf_on()
time.sleep(0.25)

try:
    BOARD.control.start_readout(**READOUT_PARAMS)

    for i in range(N_CYCLES):
        BOARD.control.toggle_trigger()
        time.sleep(0.01)

finally:
    CHANNEL_INPUT.rf_off()
    time.sleep(0.25)
    BOARD.control.stop_readout()
    time.sleep(0.25)

print("Events captured:", len(acq))

In [ ]:
EVENT_IDX = 5
event = acq.parsed_event(EVENT_IDX)

fig = create_fig(figsize=(10,8))
fig.set_facecolor('white')
plot_single_event(event, show=False)
plt.grid(True, axis=["x"])
plt.xlim([0, len(event["data"][1])])
plt.xticks([*range(0, len(event["data"][1]), 32)], labels=event["window_labels"][1])
plt.xlabel("Window Labels")
plt.title(f"WIN: {WINDOWS}, LB: {LOOKBACK}, WAT: {WAT}, READOUT: {READOUTCHANNELS}, Pulse Width: {PULSE_WIDTH}, , Event: {EVENT_IDX}")